# Rootzone Predictor - pH & EC - NoRH

This folder is self-contained for prediction with the final no-RH unified 48H model. It includes the model, metadata, this notebook, and a `master.csv` input file.

How to use:
1. Open `predict_rootzone.ipynb` from this folder.
2. Edit only the setup cell below if you want a different CSV, target time, or anchor time.
3. Run all cells.
4. Read the prediction printed at the bottom.

Required files in this folder:
- `v8_unified_model_48h_no_rh_shared_model.joblib`
- `v8_unified_model_48h_no_rh_model_meta.json`
- `master.csv`

CSV requirements:
- `timestamp` column
- at least one row before the target time with both `ph` and `ec_ms`
- data going back at least 48 hours before that anchor row
- the climate, irrigation, fertilizer, and crop columns used by the no-RH training notebook

This model does not require `internal_rh_%`. The old VPD input was replaced by a climate-demand proxy from internal air temperature, internal radiation, and ET0.


In [ ]:
# ============================================================
# EDIT ONLY THIS CELL
# ============================================================

# Input data.
# None means: use master.csv from the same folder as the saved model files.
CSV_FILE = None
# CSV_FILE = 'master.csv'

# Saved model folder.
# None means: auto-detect the folder containing the final model and metadata files.
MODEL_DIR = None
# MODEL_DIR = '.'

# Prediction target time.
# None means: use the last timestamp in the CSV.
TARGET_TIME = None
# TARGET_TIME = '2025-09-21 21:10'

# Anchor time.
# None means: use the most recent row before the target with both ph and ec_ms.
ANCHOR_TIME = None
# ANCHOR_TIME = '2025-09-21 15:30'

# Keep this False for production-style use. If True, missing input columns are filled with zero.
ALLOW_MISSING_INPUT_COLUMNS = False

# The final model uses history features from the 48 hours before the anchor pH/EC measurement.
REQUIRED_HISTORY_HOURS = 48.0

# The hist_dark_recent_6h feature needs valid radiation values before the target.
DARK_6H_WINDOW_HOURS = 6.0
# ============================================================


In [ ]:
from pathlib import Path
import importlib.util
import json
import warnings

REQUIRED_PACKAGES = [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('joblib', 'joblib'),
    ('sklearn', 'scikit-learn'),
    ('xgboost', 'xgboost'),
]
missing_packages = [install_name for import_name, install_name in REQUIRED_PACKAGES if importlib.util.find_spec(import_name) is None]
if missing_packages:
    raise ModuleNotFoundError(
        'Missing Python packages needed to run the saved model: '
        + ', '.join(missing_packages)
        + '. Install the same environment used for training before running this notebook.'
    )

import numpy as np
import pandas as pd
import joblib

try:
    from sklearn.exceptions import InconsistentVersionWarning
    warnings.filterwarnings('ignore', category=InconsistentVersionWarning)
except Exception:
    pass
warnings.filterwarnings('ignore', message='.*If you are loading a serialized model.*', category=UserWarning)

MODEL_FILE_NAME = 'v8_unified_model_48h_no_rh_shared_model.joblib'
META_FILE_NAME = 'v8_unified_model_48h_no_rh_model_meta.json'

ACID_FERTS = ['Phosphoric acid[mg]-H3PO4']
SALT_FERTS = [
    'Monopotassium Phosphate[mg] -KH2PO4',
    'Potassium Chloride[mg] - KCL',
    'Kortin [mg]',
    'Ammonium Nitrate [mg] -NH4NO3',
    'Gypsum - CaSO4*2H2O [mg]',
]
CORE_SALT_FERTS = [
    'Monopotassium Phosphate[mg] -KH2PO4',
    'Potassium Chloride[mg] - KCL',
    'Ammonium Nitrate [mg] -NH4NO3',
]

REQUIRED_INPUT_COLUMNS = [
    'timestamp',
    'ph',
    'ec_ms',
    'ET0',
    'internal_air_temp_c',
    'internal_radiation',
    'irrigation_ml_current',
    'fertilization_flag',
    'fertilization_type_a_flag',
    'fertilization_type_b_flag',
    'soil_temp_pred',
    'canopy_cover',
    'days_after_planting',
    *ACID_FERTS,
    *SALT_FERTS,
]


def candidate_paths(path_like):
    if path_like is None or str(path_like).strip() == '':
        return []

    raw = Path(path_like)
    if raw.is_absolute():
        return [raw]

    cwd = Path.cwd()
    return [
        cwd / raw,
        cwd / 'scripts' / raw,
        cwd.parent / raw,
        cwd.parent / 'scripts' / raw,
    ]


def resolve_existing_path(path_like, label):
    candidates = candidate_paths(path_like)
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists():
            return candidate

    searched = '\n  - '.join(str(c.resolve()) for c in candidates) if candidates else '  - no path was provided'
    raise FileNotFoundError(f'{label} not found. Searched:\n  - {searched}')


def resolve_model_dir(path_like=None):
    cwd = Path.cwd()
    candidates = candidate_paths(path_like) if path_like not in (None, '') else [
        cwd,
        cwd / 'saved_model',
        cwd / 'scripts' / 'saved_model',
        cwd.parent / 'saved_model',
        cwd.parent / 'scripts' / 'saved_model',
    ]

    seen = set()
    checked = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(candidate)
        if (candidate / MODEL_FILE_NAME).exists() and (candidate / META_FILE_NAME).exists():
            return candidate

    searched = '\n  - '.join(str(c) for c in checked)
    raise FileNotFoundError(
        'Could not find the saved model folder. Expected both '        f'{MODEL_FILE_NAME} and {META_FILE_NAME}. Searched:\n  - {searched}'
    )


print('Setup ready')


In [ ]:
# Feature extraction: same final feature logic used by the NoRH 48H training notebook.

def _to_num(s, default=0.0):
    return pd.to_numeric(s, errors='coerce').fillna(default)


def _sum_avail(frame, cols):
    available = [c for c in cols if c in frame.columns]
    if not available:
        return 0.0
    return float(frame[available].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum().sum())


def _sum_avail_series(frame, cols):
    available = [c for c in cols if c in frame.columns]
    if not available:
        return pd.Series(0.0, index=frame.index)
    return frame[available].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1)


def _get_fert_any(frame):
    if 'fertilization_flag' in frame.columns:
        return _to_num(frame['fertilization_flag'])
    has_a = 'fertilization_type_a_flag' in frame.columns
    has_b = 'fertilization_type_b_flag' in frame.columns
    if has_a or has_b:
        flag_a = _to_num(frame['fertilization_type_a_flag']) if has_a else pd.Series(0.0, index=frame.index)
        flag_b = _to_num(frame['fertilization_type_b_flag']) if has_b else pd.Series(0.0, index=frame.index)
        return ((flag_a > 0) | (flag_b > 0)).astype(float)
    return pd.Series(0.0, index=frame.index)


def _segment(master_df, start, stop):
    if pd.Timestamp(start) >= pd.Timestamp(stop):
        return master_df.iloc[0:0]
    return master_df.loc[(master_df.index >= pd.Timestamp(start)) & (master_df.index < pd.Timestamp(stop))]


def get_window_features(master_df, anchor_idx, current_idx):
    t0 = pd.Timestamp(anchor_idx)
    t1 = pd.Timestamp(current_idx)
    ph0 = float(master_df.loc[t0, 'ph'])
    ec0 = float(master_df.loc[t0, 'ec_ms'])
    gap_h = float((t1 - t0).total_seconds() / 3600.0)
    safe_gap_h = max(gap_h, 0.16)
    seg = _segment(master_df, t0, t1)

    irr_s = _to_num(seg['irrigation_ml_current']) if 'irrigation_ml_current' in seg.columns else pd.Series(0.0, index=seg.index)
    irr_total = float(irr_s.sum()) if len(seg) else 0.0
    fert_salt_total = _sum_avail(seg, SALT_FERTS)
    h3po4_total = _sum_avail(seg, ACID_FERTS)

    ET0_sum = float(_to_num(seg['ET0']).sum()) if 'ET0' in seg.columns else 0.0
    ET0_per_hour = float(ET0_sum / safe_gap_h)
    salt_conc_t0_t1 = float(fert_salt_total / (irr_total + 1.0))
    irr_to_et0 = float(irr_total / (ET0_sum + 0.1))

    if 'internal_air_temp_c' in seg.columns and len(seg) > 0:
        temp_s = _to_num(seg['internal_air_temp_c'])
        es_s = 0.6108 * np.exp((17.27 * temp_s) / (temp_s + 237.3))
        rad_s_tmp = _to_num(seg['internal_radiation']) if 'internal_radiation' in seg.columns else pd.Series(0.0, index=seg.index)
        rad_per_hour_tmp = float(rad_s_tmp.sum() / safe_gap_h) if len(rad_s_tmp) > 0 else 0.0
        climate_demand_proxy = float(
            es_s.mean() * (0.35 + np.log1p(max(rad_per_hour_tmp, 0.0)) / 8.0)
            + 0.10 * ET0_per_hour
        )
    else:
        temp_s = pd.Series(dtype=float)
        climate_demand_proxy = 0.0

    soil_temp_mean = float(_to_num(seg['soil_temp_pred']).mean()) if 'soil_temp_pred' in seg.columns and len(seg) > 0 else 0.0
    canopy = float(_to_num(seg['canopy_cover']).mean()) if 'canopy_cover' in seg.columns and len(seg) > 0 else 0.0
    climate_demand_pull = float(climate_demand_proxy * canopy)
    climate_demand_soil = float(climate_demand_proxy * soil_temp_mean)
    temp_x_canopy = float(soil_temp_mean * canopy)
    temp_trend = float(temp_s.iloc[-1] - temp_s.iloc[0]) if len(temp_s) > 1 else 0.0

    rad_s = _to_num(seg['internal_radiation']) if 'internal_radiation' in seg.columns else pd.Series(dtype=float)
    rad_morning = (
        float(_to_num(seg.loc[(seg.index.hour >= 6) & (seg.index.hour < 12), 'internal_radiation']).sum())
        if len(seg) > 0 and 'internal_radiation' in seg.columns
        else 0.0
    )

    fert_any = _get_fert_any(seg) if len(seg) > 0 else pd.Series(dtype=float)
    fert_times = fert_any[fert_any > 0].index
    hrs_since_fert = float((t1 - fert_times.max()).total_seconds() / 3600.0) if len(fert_times) > 0 else gap_h

    core_salt_s = _sum_avail_series(seg, CORE_SALT_FERTS) if len(seg) > 0 else pd.Series(0.0, index=seg.index)
    salt_event_idx = core_salt_s[core_salt_s > 0].index.max() if len(core_salt_s[core_salt_s > 0]) else None
    last_salt_dose = float(core_salt_s.loc[salt_event_idx]) if salt_event_idx is not None else 0.0
    last_irr_amount = float(irr_s.loc[salt_event_idx]) if salt_event_idx is not None and salt_event_idx in irr_s.index else 0.0
    hrs_since_last_salt_event = float((t1 - salt_event_idx).total_seconds() / 3600.0) if salt_event_idx is not None else gap_h
    last_event_salt_conc = float(last_salt_dose / (last_irr_amount + 1.0))
    irr_after_last_salt = (
        float(_to_num(seg.loc[seg.index > salt_event_idx, 'irrigation_ml_current']).sum())
        if salt_event_idx is not None and 'irrigation_ml_current' in seg.columns
        else 0.0
    )

    hour_b = t1.hour + t1.minute / 60.0
    days_after_planting_t1 = float(master_df.loc[t1, 'days_after_planting']) if 'days_after_planting' in master_df.columns else 0.0

    t1_soil = (
        _to_num(master_df.loc[t1:t1, 'soil_temp_pred']).iloc[0]
        if t1 in master_df.index and master_df.loc[t1, 'soil_temp_pred'] != 0
        else float(_to_num(seg['soil_temp_pred']).iloc[-1]) if len(seg) > 0 and 'soil_temp_pred' in seg.columns else 0.0
    )
    t0_soil = float(_to_num(master_df.loc[t0:t0, 'soil_temp_pred']).iloc[0]) if 'soil_temp_pred' in master_df.columns else 0.0
    rad_t1 = (
        float(_to_num(master_df.loc[t1:t1, 'internal_radiation']).iloc[0])
        if t1 in master_df.index and 'internal_radiation' in master_df.columns
        else 0.0
    )

    return {
        'ph0': ph0,
        'ec0': ec0,
        'gap_hours': gap_h,
        'irr_total_t0_t1': irr_total,
        'fert_salt_total_t0_t1': fert_salt_total,
        'ET0_sum_t0_t1': ET0_sum,
        'ET0_per_hour': ET0_per_hour,
        'h3po4_total': h3po4_total,
        'log_ec_drive': float(np.log((fert_salt_total + h3po4_total) / (irr_total + 1.0) + 0.01) - np.log(ec0 + 0.01)),
        'soil_temp_mean': soil_temp_mean,
        'climate_demand_pull': climate_demand_pull,
        'climate_demand_soil': climate_demand_soil,
        'temp_x_canopy': temp_x_canopy,
        'salt_conc_t0_t1': salt_conc_t0_t1,
        'irr_to_et0': irr_to_et0,
        'stage_x_salt_conc': float((days_after_planting_t1 / 100.0) * salt_conc_t0_t1),
        'ec_log_anchor': float(np.log(ec0 + 0.05)),
        'hrs_since_fert': hrs_since_fert,
        'hrs_since_last_salt_event': hrs_since_last_salt_event,
        'last_event_salt_conc': last_event_salt_conc,
        'irr_after_last_salt': irr_after_last_salt,
        'temp_trend': temp_trend,
        'hour_sin_b': float(np.sin(2 * np.pi * hour_b / 24.0)),
        'hour_cos_b': float(np.cos(2 * np.pi * hour_b / 24.0)),
        't1_morning_short': int((5 <= t1.hour <= 10) and (gap_h <= 1.5)),
        't1_early_day': int(5 <= hour_b < 13.0),
        'soil_delta': float(t1_soil - t0_soil),
        'rad_t1_log': float(np.log1p(rad_t1)),
        'rad_morning': rad_morning,
    }


def get_history_features(master_df, t0, t1):
    t1 = pd.Timestamp(t1)

    recent_6h = _segment(master_df, t1 - pd.Timedelta(hours=6), t1)
    prior_12_24h = _segment(master_df, t1 - pd.Timedelta(hours=24), t1 - pd.Timedelta(hours=12))
    hist24 = _segment(master_df, t1 - pd.Timedelta(hours=24), t1)

    def _irr(seg):
        return float(_to_num(seg['irrigation_ml_current']).sum()) if 'irrigation_ml_current' in seg.columns and len(seg) > 0 else 0.0

    hist_irr_recent = _irr(recent_6h)
    hist_irr_prior = _irr(prior_12_24h)
    hist_dark_recent_6h = (
        float((_to_num(recent_6h['internal_radiation']) <= 10).sum()) / 6.0
        if 'internal_radiation' in recent_6h.columns and len(recent_6h) > 0
        else 0.0
    )

    hist_acid_decay = 0.0
    if len(hist24) > 0:
        acid_doses = _sum_avail_series(hist24, ACID_FERTS)
        acid_mask = acid_doses > 0
        if acid_mask.any():
            hrs_before_t1 = np.array((t1 - hist24.index[acid_mask]).total_seconds()) / 3600.0
            hist_acid_decay = float((acid_doses[acid_mask].values * np.exp(-0.34 * hrs_before_t1)).sum())

    if len(hist24) > 1:
        hist_irr_s = _to_num(hist24['irrigation_ml_current']) if 'irrigation_ml_current' in hist24.columns else pd.Series(0.0, index=hist24.index)
        salt_s = _sum_avail_series(hist24, SALT_FERTS)
        salt_mask = salt_s > 0
        if salt_mask.any():
            hrs_before_t1 = np.array((t1 - hist24.index[salt_mask]).total_seconds()) / 3600.0
            decayed_salt = float((salt_s[salt_mask].values * np.exp(-0.15 * hrs_before_t1)).sum())
        else:
            decayed_salt = 0.0
        hist_salt_buildup = float(decayed_salt - float(hist_irr_s.sum()) * 0.1)

        fert_any = _get_fert_any(hist24)
        fert_times = hist24.index[fert_any > 0]
        hist_hrs_since_fert = float((t1 - fert_times[-1]).total_seconds() / 3600.0) if len(fert_times) > 0 else 24.0
    else:
        hist_salt_buildup = 0.0
        hist_hrs_since_fert = 24.0

    return {
        'hist_irr_recent': hist_irr_recent,
        'hist_irr_prior': hist_irr_prior,
        'hist_dark_recent_6h': hist_dark_recent_6h,
        'hist_acid_decay': hist_acid_decay,
        'hist_salt_buildup': hist_salt_buildup,
        'hist_hrs_since_fert': hist_hrs_since_fert,
    }


def get_target_prevday_features(master_df, t1):
    t1 = pd.Timestamp(t1)
    prevday = _segment(master_df, t1 - pd.Timedelta(hours=48), t1 - pd.Timedelta(hours=24))
    return {
        'hist48_irr_prevday': (
            float(_to_num(prevday['irrigation_ml_current']).sum())
            if 'irrigation_ml_current' in prevday.columns and len(prevday) > 0
            else 0.0
        )
    }




def get_features_shared(master_df, anchor_idx, current_idx):
    feats = {
        **get_window_features(master_df, anchor_idx, current_idx),
        **get_history_features(master_df, t0=anchor_idx, t1=current_idx),
        **get_target_prevday_features(master_df, t1=current_idx),
    }

    t0 = pd.Timestamp(anchor_idx)
    t1 = pd.Timestamp(current_idx)

    # Pre-anchor helpers: irrigation and decay-weighted salt before the anchor (no labeled EC needed)
    def _pre_irr(seg):
        return float(_to_num(seg['irrigation_ml_current']).sum()) \
            if 'irrigation_ml_current' in seg.columns and len(seg) > 0 else 0.0

    def _pre_salt_dw(seg, t_ref):
        if len(seg) == 0:
            return 0.0
        salt_cols = [c for c in SALT_FERTS if c in seg.columns]
        if not salt_cols:
            return 0.0
        s = seg[salt_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
        sm = s > 0
        if not sm.any():
            return 0.0
        hrs_ago = np.array((pd.Timestamp(t_ref) - seg.index[sm]).total_seconds()) / 3600.0
        return float((s[sm].values * np.exp(-0.15 * hrs_ago)).sum())

    pre6h_seg  = _segment(master_df, t0 - pd.Timedelta(hours=6),  t0)
    pre24h_seg = _segment(master_df, t0 - pd.Timedelta(hours=24), t0)
    irr_6h   = _pre_irr(pre6h_seg)
    irr_24h  = _pre_irr(pre24h_seg)
    salt_6h  = _pre_salt_dw(pre6h_seg,  t0)
    salt_24h = _pre_salt_dw(pre24h_seg, t0)

    # Pre-anchor salt/irrigation ratio + pre-anchor salt buildup
    pre_salt_irr_ratio_6h  = float(salt_6h  / (irr_6h  + 1.0))
    pre_salt_irr_ratio_24h = float(salt_24h / (irr_24h + 1.0))

    # Decay-weighted salt buildup in 24h before the anchor (t0-referenced)
    hist24_t0 = _segment(master_df, t0 - pd.Timedelta(hours=24), t0)
    if len(hist24_t0) > 1:
        hi_s = _to_num(hist24_t0['irrigation_ml_current']) \
            if 'irrigation_ml_current' in hist24_t0.columns else pd.Series(0.0, index=hist24_t0.index)
        pre_anchor_salt_buildup = float(_pre_salt_dw(hist24_t0, t0) - float(hi_s.sum()) * 0.1)
    else:
        pre_anchor_salt_buildup = 0.0

    gap_h_safe = max(float(feats['gap_hours']), 0.25)
    hist48_pre_t0 = _segment(master_df, t0 - pd.Timedelta(hours=48), t0)
    salt48_pre_t0 = float(_sum_avail(hist48_pre_t0, SALT_FERTS)) if len(hist48_pre_t0) > 0 else 0.0

    positive_pre_anchor_salt_buildup = max(pre_anchor_salt_buildup, 0.0)
    salt_carryover_pressure = float(positive_pre_anchor_salt_buildup * np.exp(-gap_h_safe / 8.0))
    ec_salt_carryover_pressure = float(feats['ec0'] * np.log1p(positive_pre_anchor_salt_buildup) * np.exp(-gap_h_safe / 8.0))

    feats.update({
        'pre_salt_irr_ratio_6h':   pre_salt_irr_ratio_6h,
        'pre_salt_irr_ratio_24h':  pre_salt_irr_ratio_24h,
        'pre_anchor_salt_buildup': pre_anchor_salt_buildup,
        'salt_carryover_pressure': salt_carryover_pressure,
        'ec_salt_carryover_pressure': ec_salt_carryover_pressure,
        'high_ec_salt_carryover': int(feats['ec0'] >= 2.0 and positive_pre_anchor_salt_buildup >= 250.0 and feats['gap_hours'] <= 3.0),
        'salt_recency_pressure': float(feats['last_event_salt_conc'] * np.exp(-feats['hrs_since_last_salt_event'] / 6.0)),
        'salt_event_pos': float(np.clip(1.0 - (feats['hrs_since_last_salt_event'] / gap_h_safe), 0.0, 1.0)),
        'ec0_x_p48_salt': float(feats['ec0'] * salt48_pre_t0),
    })
    return feats



def extract_features(df, anchor_ts, target_ts):
    return get_features_shared(df, anchor_ts, target_ts)


def robust_linear_gate(feats, meta):
    return bool(
        feats.get('fert_salt_total_t0_t1', 0.0) >= 250.0 and
        feats.get('irr_after_last_salt', 0.0) <= 1.0 and
        feats.get('salt_conc_t0_t1', 0.0) >= 3.0 and
        feats.get('ec0', np.inf) <= 0.8 and
        feats.get('gap_hours', np.inf) <= meta['ROBUST_LINEAR_GATE_MAX_GAP_H']
    )


def predict_rootzone(feats, model, meta):
    feature_cols = list(meta['feature_cols'])
    robust_feature_cols = list(meta.get('robust_feature_cols', feature_cols))
    required_features = list(dict.fromkeys(feature_cols + robust_feature_cols))
    missing_features = [f for f in required_features if f not in feats]
    if missing_features:
        raise ValueError(f'Missing model features: {missing_features}')

    X = pd.DataFrame([feats])[feature_cols]
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    base_raw = np.asarray(model['base'].predict(X))[0]
    base_raw = base_raw * model['target_std'] + model['target_mean']

    model_component = 'xgboost'
    final_raw = base_raw
    robust_linear = model.get('robust_linear')

    if robust_linear is not None and robust_linear_gate(feats, meta):
        X_robust = pd.DataFrame([feats])[robust_feature_cols]
        X_robust = X_robust.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        robust_raw = np.asarray(robust_linear.predict(X_robust))[0]
        robust_raw = robust_raw * model['target_std'] + model['target_mean']
        blend = np.array([meta['ROBUST_LINEAR_BLEND_PH'], meta['ROBUST_LINEAR_BLEND_EC']], dtype=float)
        final_raw = (1.0 - blend) * base_raw + blend * robust_raw
        model_component = 'robust_linear'

    ph_pred = feats['ph0'] + final_raw[0]
    ec_pred = max(0.0, (feats['ec0'] + meta['EC_TARGET_SHIFT']) * np.exp(final_raw[1]) - meta['EC_TARGET_SHIFT'])

    return {
        'ph_pred': float(ph_pred),
        'ec_pred': float(ec_pred),
        'model_component': model_component,
    }


print('Feature and prediction functions ready')


In [ ]:
model_dir = resolve_model_dir(MODEL_DIR)
model_path = model_dir / MODEL_FILE_NAME
meta_path = model_dir / META_FILE_NAME

if not model_path.exists():
    raise FileNotFoundError(f'Model file not found: {model_path}')
if not meta_path.exists():
    raise FileNotFoundError(f'Metadata file not found: {meta_path}')

model = joblib.load(model_path)
with open(meta_path, 'r', encoding='utf-8') as f:
    meta = json.load(f)

required_model_keys = ['base', 'target_mean', 'target_std']
missing_model_keys = [k for k in required_model_keys if k not in model]
if missing_model_keys:
    raise ValueError(f'Model artifact is missing required fields: {missing_model_keys}')

required_meta_keys = [
    'feature_cols',
    'EC_TARGET_SHIFT',
    'ROBUST_LINEAR_BLEND_PH',
    'ROBUST_LINEAR_BLEND_EC',
    'ROBUST_LINEAR_GATE_MAX_GAP_H',
    'SHARED_SKIP_MAX_GAP_H',
]
missing_meta_keys = [k for k in required_meta_keys if k not in meta]
if missing_meta_keys:
    raise ValueError(f'Model metadata is missing required fields: {missing_meta_keys}')
if 'prev_ec_slope' in meta['feature_cols']:
    raise ValueError('Loaded metadata belongs to an older model version. Use the final no-RH 48H model files.')
if not meta.get('NO_INTERNAL_RH', False):
    raise ValueError('Loaded metadata is not marked as a no-RH model. Use the no-RH 48H model files.')

if CSV_FILE is None or str(CSV_FILE).strip() == '':
    csv_path = model_dir / 'master.csv'
    if not csv_path.exists():
        raise FileNotFoundError(f'Default CSV not found next to model files: {csv_path}')
else:
    csv_path = resolve_existing_path(CSV_FILE, 'CSV file')
raw_df = pd.read_csv(csv_path)
if 'timestamp' not in raw_df.columns:
    raise ValueError('CSV must contain a timestamp column')
raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
if raw_df['timestamp'].isna().any():
    raise ValueError('CSV contains timestamp values that could not be parsed')

missing_columns = [c for c in REQUIRED_INPUT_COLUMNS if c not in raw_df.columns]
if missing_columns:
    if not ALLOW_MISSING_INPUT_COLUMNS:
        missing_text = '\n'.join(f'  - {c}' for c in missing_columns)
        raise ValueError(
            'Missing input columns needed to run the model as trained:\n'
            + missing_text
            + '\n\nSet ALLOW_MISSING_INPUT_COLUMNS=True only if you intentionally want missing columns filled with zero.'
        )
    for col in missing_columns:
        raw_df[col] = 0.0

raw_df = raw_df.sort_values('timestamp')
raw_df = raw_df.drop_duplicates(subset='timestamp', keep='last')
df = raw_df.set_index('timestamp')
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

if TARGET_TIME is None:
    target_ts = df.index.max()
else:
    target_ts = pd.Timestamp(TARGET_TIME)
    if target_ts not in df.index:
        placeholder = pd.DataFrame(index=[target_ts], columns=df.columns, dtype=float)
        df = pd.concat([df, placeholder]).sort_index()

if ANCHOR_TIME is None:
    labeled_before_target = df.loc[df.index < target_ts]
    labeled_before_target = labeled_before_target[labeled_before_target['ph'].notna() & labeled_before_target['ec_ms'].notna()]
    if labeled_before_target.empty:
        raise ValueError('No anchor row found before the target time with both ph and ec_ms filled in')
    anchor_ts = labeled_before_target.index[-1]
else:
    anchor_ts = pd.Timestamp(ANCHOR_TIME)
    if anchor_ts not in df.index:
        raise ValueError(f'ANCHOR_TIME is not present in the CSV: {anchor_ts}')
    if pd.isna(df.loc[anchor_ts, 'ph']) or pd.isna(df.loc[anchor_ts, 'ec_ms']):
        raise ValueError('ANCHOR_TIME row must contain both ph and ec_ms')

gap_hours = (target_ts - anchor_ts).total_seconds() / 3600.0
if gap_hours <= 0:
    raise ValueError('Target time must be after the anchor time')

required_history_start = anchor_ts - pd.Timedelta(hours=REQUIRED_HISTORY_HOURS)
if df.index.min() > required_history_start:
    available_history_h = (anchor_ts - df.index.min()).total_seconds() / 3600.0
    raise ValueError(
        f'The model needs {REQUIRED_HISTORY_HOURS:.0f}h of data before the anchor. '
        f'This CSV only has {available_history_h:.1f}h before {anchor_ts:%Y-%m-%d %H:%M}. '
        'Add earlier greenhouse rows or choose a later anchor time.'
    )

history_window = df.loc[(df.index >= required_history_start) & (df.index < anchor_ts)]
if history_window.empty:
    raise ValueError('No input rows found in the required 48h history window before the anchor')

dark6_start = target_ts - pd.Timedelta(hours=DARK_6H_WINDOW_HOURS)
dark6_window = df.loc[(df.index >= dark6_start) & (df.index < target_ts)]
if dark6_window.empty:
    raise ValueError(
        f'No rows found in the {DARK_6H_WINDOW_HOURS:.0f}h radiation window before the target. '
        'The hist_dark_recent_6h feature needs internal_radiation data before the target time.'
    )
if 'internal_radiation' not in dark6_window.columns:
    raise ValueError('internal_radiation is required for the hist_dark_recent_6h feature')
dark6_radiation = pd.to_numeric(dark6_window['internal_radiation'], errors='coerce')
if dark6_radiation.notna().sum() == 0:
    raise ValueError(
        f'internal_radiation has no valid numeric values in the {DARK_6H_WINDOW_HOURS:.0f}h window before the target. '
        'The hist_dark_recent_6h feature cannot be calculated reliably.'
    )
if dark6_radiation.isna().any():
    missing_n = int(dark6_radiation.isna().sum())
    raise ValueError(
        f'internal_radiation has {missing_n} missing/non-numeric values in the {DARK_6H_WINDOW_HOURS:.0f}h window before the target. '
        'Fill or remove those rows before running prediction so missing values are not treated as darkness.'
    )

max_gap_h = float(meta.get('SHARED_SKIP_MAX_GAP_H', 48.0))
if gap_hours > max_gap_h:
    raise ValueError(
        f'Target is {gap_hours:.1f}h after the anchor, but the model is intended for <= {max_gap_h:.0f}h predictions. '
        'Choose a closer target or provide a newer pH/EC anchor row.'
    )

print(f'Model loaded: {model_path.name}')
print(f'Metadata loaded: {meta_path.name}')
print(f'CSV loaded: {csv_path.name} ({len(df)} rows)')
print(f'Anchor: {anchor_ts:%Y-%m-%d %H:%M}  pH={df.loc[anchor_ts, "ph"]:.3f}  EC={df.loc[anchor_ts, "ec_ms"]:.4f}')
print(f'Target: {target_ts:%Y-%m-%d %H:%M}  gap={gap_hours:.1f}h')
print(f'History: {REQUIRED_HISTORY_HOURS:.0f}h before anchor available')
print(f'Dark-window radiation: {len(dark6_radiation)} samples checked over {DARK_6H_WINDOW_HOURS:.0f}h before target')
print('Input checks passed')


In [ ]:
feats = extract_features(df, anchor_ts, target_ts)
result = predict_rootzone(feats, model, meta)

print()
print('=' * 54)
print('ROOTZONE PREDICTION')
print('=' * 54)
print(f'Anchor time : {anchor_ts:%Y-%m-%d %H:%M}')
print(f'Anchor pH   : {feats["ph0"]:.3f}')
print(f'Anchor EC   : {feats["ec0"]:.4f} mS/cm')
print(f'Target time : {target_ts:%Y-%m-%d %H:%M}')
print(f'Gap         : {feats["gap_hours"]:.1f} hours')
print(f'Component   : {result["model_component"]}')
print('-' * 54)
print(f'Predicted pH : {result["ph_pred"]:.3f}')
print(f'Predicted EC : {result["ec_pred"]:.4f} mS/cm')
print('=' * 54)
